# 지자체복지서비스 API 수집 — 전북·서울·경기 (목록 + 상세)

이 노트북은 **전북특별자치도, 서울특별시, 경기도**의 복지서비스를 다음 순서로 수집합니다.

1. 시·도별 목록조회 → `df_jeonbuk`, `df_seoul`, `df_gyeonggi`, `df_all`
2. 목록에서 확보한 `servId`로 상세조회 (`callTp=D`)
3. 상세 응답과 목록 병합 → **`df_full` 및 지역별 `df_jeonbuk_full`, `df_seoul_full`, `df_gyeonggi_full`**
4. CSV / JSONL 저장, 상세조회 성공·실패 현황 확인

`.env`의 API 키를 사용합니다. **실제 API 응답은 키를 가진 사용자 환경에서 노트북 실행 시 수집**됩니다. 상세조회는 서비스별 1회 호출이어서 호출량 제한에 유의하세요.

## 0. 준비

노트북과 같은 폴더에 `.env` 파일을 준비합니다.

```dotenv
DATA_GO_KR_API_KEY=발급받은_디코딩_인증키
```

기존 환경변수명이 다르면 아래 `KEY_ENV_NAMES`에 추가하면 됩니다. `.env`를 Git에 올리지 마세요 (`.gitignore`에 `.env` 추가).

In [1]:
%pip -q install requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Users\User\OneDrive\Desktop\SKN 부트캠프\3rd-project\.venv\Scripts\python.exe: No module named pip


## 1. 라이브러리 및 API 주소

In [2]:
import os
import time
import math
from pathlib import Path
from datetime import datetime, timezone, timedelta
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

load_dotenv(override=True)

KEY_ENV_NAMES = ["DATA_GO_KR_API_KEY", "DATA_GO_KR_SERVICE_KEY", "SERVICE_KEY"]
SERVICE_KEY = next((os.getenv(k) for k in KEY_ENV_NAMES if os.getenv(k)), None)
if not SERVICE_KEY:
    raise ValueError(f".env에 인증키가 없습니다. 확인할 변수명: {KEY_ENV_NAMES}")

BASE_URL = "https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations"
API_URLS = {
    "list": f"{BASE_URL}/LcgvWelfarelist",
    "detail": f"{BASE_URL}/LcgvWelfaredetailed",
}
REGIONS = ["전북특별자치도", "서울특별시", "경기도"]
PAGE_SIZE = 100
REQUEST_DELAY = 0.3
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("인증키 로드 성공 (값은 표시하지 않음)")
print("대상 지역:", ", ".join(REGIONS))
print("목록조회 URL:", API_URLS["list"])
print("상세조회 URL:", API_URLS["detail"])

인증키 로드 성공 (값은 표시하지 않음)
대상 지역: 전북특별자치도, 서울특별시, 경기도
목록조회 URL: https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations/LcgvWelfarelist
상세조회 URL: https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations/LcgvWelfaredetailed


## 2. API 호출 함수

`ctpvNm`으로 시·도를 지정하고 `sggNm`은 넣지 않아 시·도 전체를 검색합니다. XML의 `resultCode=0` 여부를 검사합니다. 오류가 나면 해당 지역 수집이 중단되므로, 불완전한 결과가 정상 완료로 오인되지 않습니다.

In [3]:
retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))


def fetch_page(region: str, page_no: int, page_size: int = PAGE_SIZE):
    params = {
        "serviceKey": SERVICE_KEY,
        "callTp": "L",
        "ctpvNm": region,
        "pageNo": page_no,
        "numOfRows": page_size,
    }
    res = session.get(API_URLS["list"], params=params, timeout=30)
    res.raise_for_status()
    try:
        root = ET.fromstring(res.content)
    except ET.ParseError as exc:
        raise RuntimeError(f"{region} {page_no}페이지: XML 파싱 실패") from exc

    code = root.findtext("resultCode")
    message = root.findtext("resultMessage", "")
    if code != "0":
        raise RuntimeError(f"{region} {page_no}페이지: API 오류(code={code}, message={message})")

    total = int(root.findtext("totalCount", "0"))
    rows = [
        {child.tag: child.text for child in item}
        for item in root.findall("servList")
    ]
    return total, rows


def fetch_region(region: str, page_size: int = PAGE_SIZE) -> pd.DataFrame:
    total, first_rows = fetch_page(region, 1, page_size)
    pages = math.ceil(total / page_size) if total else 0
    all_rows = list(first_rows)
    print(f"[{region}] API 조회 총 {total:,}건 / 예상 {pages}페이지")

    for page in range(2, pages + 1):
        time.sleep(REQUEST_DELAY)
        page_total, rows = fetch_page(region, page, page_size)
        if page_total != total:
            print(f"  ⚠ 전체 건수 변경: 최초 {total} → {page_total} (수집 중 갱신 가능)")
        if not rows:
            raise RuntimeError(f"{region} {page}페이지가 비어 있습니다. 페이지 제한/응답 확인 필요")
        all_rows.extend(rows)
        print(f"  {page}/{pages} 페이지 완료, 누적 {len(all_rows):,}건")

    df = pd.DataFrame(all_rows)
    if df.empty and total != 0:
        raise RuntimeError(f"{region}: totalCount={total}인데 결과가 비어 있습니다")
    if len(df) != total:
        raise RuntimeError(
            f"{region}: API totalCount={total}, 실제 수집={len(df)}. "
            "페이지 크기 제한 또는 수집 중 데이터 갱신 여부를 확인하세요."
        )
    if not df.empty:
        df["requested_region"] = region
    print(f"[{region}] 수집 완료: {len(df):,}건\n")
    return df

## 3. 전북·서울·경기 전체 목록 수집

이 셀을 실행하면 실시간으로 API를 호출합니다. 첫 지역에서 오류가 나면 인증키·활용신청 상태와 `.env`를 확인하세요. `PAGE_SIZE=100`이 서버에서 허용되지 않는 경우 10 또는 50으로 낮춰 재실행하세요.

In [4]:
region_dfs = {}

for region in REGIONS:
    region_dfs[region] = fetch_region(region)

# 지역별 DataFrame
df_jeonbuk = region_dfs["전북특별자치도"]
df_seoul = region_dfs["서울특별시"]
df_gyeonggi = region_dfs["경기도"]

# 통합 DataFrame (지역별 결과 원본을 그대로 합침)
df_all = pd.concat(
    [df_jeonbuk, df_seoul, df_gyeonggi],
    ignore_index=True,
    sort=False,
)

print("지역별 수집 건수")
display(pd.DataFrame({"지역": list(region_dfs), "수집 건수": [len(x) for x in region_dfs.values()]}))
print(f"전체 행 수: {len(df_all):,}")
display(df_all.head())

[전북특별자치도] API 조회 총 396건 / 예상 4페이지
  2/4 페이지 완료, 누적 200건
  3/4 페이지 완료, 누적 300건
  4/4 페이지 완료, 누적 396건
[전북특별자치도] 수집 완료: 396건

[서울특별시] API 조회 총 397건 / 예상 4페이지
  2/4 페이지 완료, 누적 200건
  3/4 페이지 완료, 누적 300건
  4/4 페이지 완료, 누적 397건
[서울특별시] 수집 완료: 397건

[경기도] API 조회 총 692건 / 예상 7페이지
  2/7 페이지 완료, 누적 200건
  3/7 페이지 완료, 누적 300건
  4/7 페이지 완료, 누적 400건
  5/7 페이지 완료, 누적 500건
  6/7 페이지 완료, 누적 600건
  7/7 페이지 완료, 누적 692건
[경기도] 수집 완료: 692건

지역별 수집 건수


,지역,수집 건수
0,전북특별자치도,396
1,서울특별시,397
2,경기도,692


전체 행 수: 1,485


,aplyMtdNm,bizChrDeptNm,ctpvNm,inqNum,intrsThemaNmArray,lastModYmd,servDgst,servDtlLink,servId,servNm,sggNm,sprtCycNm,srvPvsnNm,trgterIndvdlNmArray,lifeNmArray,requested_region
0,방문,전북특별자치도 순창군 행정복지국 건강장수과,전북특별자치도,2590,서민금융,20261028,관내 화장시설의 부재로 타 지역의 화장장 사용에 대한 군민의 경제적인 부담 해소와 ...,https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfa...,WLF00002438,군민 화장장려금 지원 사업,순창군,수시,현금지급,NaN,NaN,전북특별자치도
1,"방문, 전화",전북특별자치도 남원시 자치행정국 주민복지과,전북특별자치도,13574,"신체건강, 안전·위기",20261014,저소득층의 간병비 부담을 완화하고 입원환자의 안정적인 치료 및 회복을 지원,https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfa...,WLF00001316,무료간병 온종일서비스,남원시,1회성,프로그램/서비스(서비스),저소득,NaN,전북특별자치도
2,방문,전북특별자치도 무주군 행정복지국 인구활력과,전북특별자치도,3206,"서민금융, 일자리",20261013,취업취약계층에게 한시적 일자리 제공을 통한 생계 지원,https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfa...,WLF00003485,공공근로사업,무주군,월,현금지급,NaN,"청년, 노년, 중장년",전북특별자치도
3,"방문, 전화",전북특별자치도 정읍시 복지환경국 사회복지과,전북특별자치도,2437,NaN,20261013,행여자의 귀향여비 지급 및 숙박비 지급,https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfa...,WLF00000453,행여자 관리(여비 및 숙박비 지급),정읍시,수시,기타,NaN,"노년, 중장년, 청년",전북특별자치도
4,방문,전북특별자치도 임실군 복지환경국 주민복지과,전북특별자치도,3190,NaN,20261013,수급자가정 방문 방역사업을 실시하여 쾌적한 주거공간을 조성하고 각종 전염병 \r\n...,https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfa...,WLF00002346,수급자가정 방문방역사업,임실군,반기,프로그램/서비스(서비스),저소득,NaN,전북특별자치도


## 4. 검증

같은 서비스 ID가 지역에 중복 나타날 수 있으므로 **원본 전체 행은 삭제하지 않습니다.** 중복 건수를 확인하고 필요할 때만 별도 중복 제거본을 만드세요.

In [5]:
print("전체 데이터 shape:", df_all.shape)
print("컬럼 목록:", df_all.columns.tolist())

if "servId" in df_all.columns:
    print("고유 서비스 ID:", df_all["servId"].nunique())
    print("서비스 ID 중복 행:", df_all["servId"].duplicated().sum())

if "sggNm" in df_all.columns and "ctpvNm" in df_all.columns:
    print("시군구별 건수 (상위 30):")
    display(df_all.groupby(["ctpvNm", "sggNm"], dropna=False).size().rename("건수").reset_index().sort_values("건수", ascending=False).head(30))

print("누락값 상위:")
display(df_all.isna().sum().sort_values(ascending=False).head(20))

전체 데이터 shape: (1485, 16)
컬럼 목록: ['aplyMtdNm', 'bizChrDeptNm', 'ctpvNm', 'inqNum', 'intrsThemaNmArray', 'lastModYmd', 'servDgst', 'servDtlLink', 'servId', 'servNm', 'sggNm', 'sprtCycNm', 'srvPvsnNm', 'trgterIndvdlNmArray', 'lifeNmArray', 'requested_region']
고유 서비스 ID: 1485
서비스 ID 중복 행: 0
시군구별 건수 (상위 30):


,ctpvNm,sggNm,건수
33,경기도,NaN,81
61,서울특별시,NaN,57
77,전북특별자치도,NaN,45
63,전북특별자치도,군산시,38
13,경기도,성남시,38
65,전북특별자치도,남원시,33
70,전북특별자치도,익산시,29
75,전북특별자치도,정읍시,29
5,경기도,광명시,29
16,경기도,안산시,28


누락값 상위:


trgterIndvdlNmArray    793
lifeNmArray            508
intrsThemaNmArray      393
aplyMtdNm              300
sggNm                  183
srvPvsnNm                1
ctpvNm                   0
bizChrDeptNm             0
servDtlLink              0
servDgst                 0
lastModYmd               0
inqNum                   0
sprtCycNm                0
servNm                   0
servId                   0
requested_region         0
dtype: int64

## 5. CSV 저장

Excel에서 한글이 잘 보이도록 `utf-8-sig` 인코딩으로 저장합니다. 날짜를 포함한 파일명이므로 이전 수집 결과가 덮어써지지 않습니다.

In [6]:
date_tag = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
file_map = {
    "jeonbuk": df_jeonbuk,
    "seoul": df_seoul,
    "gyeonggi": df_gyeonggi,
    "all": df_all,
}

saved_paths = []
for label, df_region in file_map.items():
    path = OUTPUT_DIR / f"welfare_{label}_{date_tag}.csv"
    df_region.to_csv(path, index=False, encoding="utf-8-sig")
    saved_paths.append(path)

for path in saved_paths:
    print(f"저장 완료: {path} ({path.stat().st_size:,} bytes)")

저장 완료: outputs\welfare_jeonbuk_20261107_103452.csv (208,912 bytes)
저장 완료: outputs\welfare_seoul_20261107_103452.csv (218,412 bytes)
저장 완료: outputs\welfare_gyeonggi_20261107_103452.csv (345,877 bytes)
저장 완료: outputs\welfare_all_20261107_103452.csv (772,841 bytes)


## 6. (선택) 노년 관련 데이터만 확인

목록의 `lifeNmArray`에 '노년'이 적힌 행만 추려볼 수 있습니다. **이 필드가 비었거나 일반연령 정책인 경우도 있으므로 본 데이터셋은 삭제하지 마세요.**

In [7]:
if "lifeNmArray" in df_all.columns:
    df_senior_tagged = df_all[
        df_all["lifeNmArray"].fillna("").str.contains("노년", regex=False)
    ].copy()
    print("생애주기에 '노년'이 표시된 건수:", len(df_senior_tagged))
    display(df_senior_tagged[[c for c in ["servId", "servNm", "ctpvNm", "sggNm", "lifeNmArray"] if c in df_senior_tagged.columns]].head(20))
else:
    print("lifeNmArray 필드가 없어 노년 태그 필터를 건너뜁니다.")

생애주기에 '노년'이 표시된 건수: 375


,servId,servNm,ctpvNm,sggNm,lifeNmArray
2,WLF00003485,공공근로사업,전북특별자치도,무주군,"청년, 노년, 중장년"
3,WLF00000453,행여자 관리(여비 및 숙박비 지급),전북특별자치도,정읍시,"노년, 중장년, 청년"
11,WLF00006140,지역봉사지도원 운영,전북특별자치도,완주군,노년
15,WLF00004140,미등록 경로당 운영 지원 사업,전북특별자치도,장수군,노년
20,WLF00003795,효도수당 지원,전북특별자치도,진안군,노년
28,WLF00000557,임실군 노부모 모시는 가정 지원,전북특별자치도,임실군,노년
30,WLF00002352,무료급식소 및 장애우 쉼터 운영비,전북특별자치도,군산시,"노년, 중장년"
37,WLF00003859,노부모 모시는 가정 효도수당 지원,전북특별자치도,장수군,노년
44,WLF00002104,보훈단체일자리사업,전북특별자치도,전주시,노년
50,WLF00000450,도시가스 공급사업 보조금 지원,전북특별자치도,익산시,"중장년, 청년, 노년"


### 참고 및 한계

- 원천: [공공데이터포털 지자체복지서비스](https://www.data.go.kr/tcs/dss/selectApiDataDetailView.do?publicDataPk=15108347)
- 조회 경로: `.../LcgvWelfarelist`, `.../LcgvWelfaredetailed`
- 상세조회는 `servId`마다 1회 요청하므로 일일 호출 한도를 확인하세요.
- 상세 결과에는 이용자별 실제 수급자격 판정이 포함되지 않으며, 최신 정책과 신청 가능 여부를 공식 사이트에서 재확인해야 합니다.
- 실시간 API 접근 없이 작성된 코드이므로 첫 1건의 상세조회 응답 구조를 검증하고 전체 실행하도록 구성했습니다.

## 7. 상세조회 함수 (`servId` → 상세 XML)

**핵심:** 목록조회에서 얻은 `servId`를 이용해 상세조회 URL에 `callTp='D'`와 `servId`를 보냅니다. 상세 XML의 기본 필드(지원대상 `sprtTrgtCn`, 선정기준 `slctCritCn`, 지원내용 `alwServCn`, 신청방법 `aplyMtdCn`)뿐 아니라 나머지 태그도 가능한 한 보존합니다. 여러 번 나타나는 태그는 JSON 문자열로 저장합니다.

In [8]:
import json
from collections import defaultdict


def _flatten_detail_xml(root):
    """API 응답 내 모든 말단 텍스트를 태그명 기준으로 수집합니다.
    동일 태그가 반복되면 JSON 배열 문자열로 저장합니다.
    """
    values = defaultdict(list)
    for node in root.iter():
        if len(node) == 0:
            value = (node.text or "").strip()
            if value and node.tag not in {"resultCode", "resultMessage"}:
                values[node.tag].append(value)
    return {
        k: vals[0] if len(vals) == 1 else json.dumps(vals, ensure_ascii=False)
        for k, vals in values.items()
    }


def fetch_detail(serv_id: str) -> dict:
    params = {
        "serviceKey": SERVICE_KEY,
        "callTp": "D",
        "servId": str(serv_id),
    }
    res = session.get(API_URLS["detail"], params=params, timeout=30)
    res.raise_for_status()
    try:
        detail_root = ET.fromstring(res.content)
    except ET.ParseError as exc:
        raise RuntimeError(f"{serv_id}: 상세 XML 파싱 실패") from exc

    # 상세조회 정상 응답의 메시지는 wantedDtl 등의 루트에 존재할 수 있습니다.
    code_value = detail_root.findtext(".//resultCode")
    message = detail_root.findtext(".//resultMessage", "")
    if code_value != "0":
        raise RuntimeError(f"{serv_id}: 상세 API 오류(code={code_value}, message={message})")

    row = _flatten_detail_xml(detail_root)
    row["servId"] = str(serv_id)
    return row


## 8. 상세조회 1건 테스트

전체 호출 전에 **단 1건**으로 상세 응답에 실제 내용이 들어오는지 확인합니다. 이 셀이 실패하면 아래 일괄 상세조회는 실행하지 마세요.

In [9]:
if df_all.empty or "servId" not in df_all.columns:
    raise ValueError("목록조회 결과가 없거나 servId 컬럼이 없습니다.")

sample_id = df_all["servId"].dropna().astype(str).iloc[0]
sample_detail = fetch_detail(sample_id)
print("테스트 서비스 ID:", sample_id)
print("상세조회 필드 수:", len(sample_detail))
display(pd.DataFrame([sample_detail]).T.rename(columns={0: "상세조회 결과"}).head(35))

# 1건 이상 실질 상세 필드가 있는지 확인
expected_detail_fields = {"sprtTrgtCn", "slctCritCn", "alwServCn", "aplyMtdCn"}
found_fields = expected_detail_fields.intersection(sample_detail)
print("대표 상세 필드:", sorted(found_fields))
if len(sample_detail) <= 2:
    raise RuntimeError("상세조회 결과에 정보가 거의 없습니다. 실제 응답 XML 확인 필요")


테스트 서비스 ID: WLF00002438
상세조회 필드 수: 21


,상세조회 결과
servId,WLF00002438
servNm,군민 화장장려금 지원 사업
enfcBgngYmd,20160801
enfcEndYmd,99991231
bizChrDeptNm,전북특별자치도 순창군 행정복지국 건강장수과
ctpvNm,전북특별자치도
sggNm,순창군
servDgst,관내 화장시설의 부재로 타 지역의 화장장 사용에 대한 군민의 경제적인 부담 해소와 ...
intrsThemaNmArray,서민금융
sprtCycNm,수시


대표 상세 필드: ['alwServCn', 'aplyMtdCn', 'slctCritCn', 'sprtTrgtCn']


## 9. 전체 상세조회 + 중간 저장(재실행 가능)

- 기본값은 **3개 지역에서 조회한 고유 `servId` 전체**입니다. API 호출 한도가 걱정되면 `DETAIL_LIMIT = 20`으로 먼저 시험하세요.
- 각 성공 건을 `outputs/welfare_detail_cache.jsonl`에 즉시 저장합니다. 노트북을 다시 실행하면 이미 성공한 ID는 재호출하지 않습니다.
- 개별 실패 건은 오류를 따로 기록하고 다음 서비스로 넘어갑니다. **실패 항목은 성공으로 처리하지 않으며** 결과 검증에서 확인합니다.
- 재수집을 원하면 캐시 파일을 다른 이름으로 이동한 다음 다시 실행하세요.

In [12]:
DETAIL_LIMIT = None  # 첫 실행에서 가볍게 시험하려면 20, 전체 수집은 None
DETAIL_DELAY = 0.35  # 호출량·초당 제한에 맞게 조정
CACHE_PATH = OUTPUT_DIR / "welfare_detail_cache.jsonl"
ERROR_PATH = OUTPUT_DIR / "welfare_detail_errors.csv"

# 기존 캐시 복원
cache = {}
if CACHE_PATH.exists():
    with CACHE_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                if obj.get("servId"):
                    cache[str(obj["servId"])] = obj
print(f"기존 상세조회 캐시: {len(cache):,}건")

service_ids = df_all["servId"].dropna().astype(str).drop_duplicates().tolist()
if DETAIL_LIMIT is not None:
    service_ids = service_ids[:DETAIL_LIMIT]

# 위 1건 테스트 결과도 캐시에 저장(이미 있다면 덮어쓰지 않음)
if sample_id in service_ids and sample_id not in cache:
    cache[sample_id] = sample_detail
    with CACHE_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(sample_detail, ensure_ascii=False) + "\n")

errors = []
target_count = len(service_ids)
for i, serv_id in enumerate(service_ids, start=1):
    if serv_id in cache:
        continue
    try:
        item = fetch_detail(serv_id)
        cache[serv_id] = item
        with CACHE_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    except Exception as exc:
        errors.append({"servId": serv_id, "error": str(exc)})
        print(f"  실패 {i}/{target_count}: {serv_id} | {exc}")
    if i % 25 == 0 or i == target_count:
        print(f"진행 {i}/{target_count} | 상세 성공 누적 {sum(k in cache for k in service_ids)} | 이번 실행 오류 {len(errors)}")
    time.sleep(DETAIL_DELAY)

df_detail = pd.DataFrame([cache[sid] for sid in service_ids if sid in cache])
df_errors = pd.DataFrame(errors, columns=["servId", "error"])
df_errors.to_csv(ERROR_PATH, index=False, encoding="utf-8-sig")
print(f"요청 대상: {target_count:,}건 / 상세 확보: {len(df_detail):,}건 / 미확보: {target_count - len(df_detail):,}건")
print("오류 목록:", ERROR_PATH)


기존 상세조회 캐시: 1,000건
  실패 1001/1485: WLF00001523 | HTTPSConnectionPool(host='apis.data.go.kr', port=443): Max retries exceeded with url: /B554287/LocalGovernmentWelfareInformations/LcgvWelfaredetailed?serviceKey=f90186c3970e1aa62bfbcce045ed19456c7fa5b05d738f555607c49b5b65d460&callTp=D&servId=WLF00001523 (Caused by ResponseError('too many 429 error responses'))
  실패 1002/1485: WLF00000941 | HTTPSConnectionPool(host='apis.data.go.kr', port=443): Max retries exceeded with url: /B554287/LocalGovernmentWelfareInformations/LcgvWelfaredetailed?serviceKey=f90186c3970e1aa62bfbcce045ed19456c7fa5b05d738f555607c49b5b65d460&callTp=D&servId=WLF00000941 (Caused by ResponseError('too many 429 error responses'))
  실패 1003/1485: WLF00003056 | HTTPSConnectionPool(host='apis.data.go.kr', port=443): Max retries exceeded with url: /B554287/LocalGovernmentWelfareInformations/LcgvWelfaredetailed?serviceKey=f90186c3970e1aa62bfbcce045ed19456c7fa5b05d738f555607c49b5b65d460&callTp=D&servId=WLF00003056 (Caused by Re

KeyboardInterrupt: 

In [13]:
df_detail = pd.DataFrame([cache[sid] for sid in service_ids if sid in cache])
df_errors = pd.DataFrame(errors, columns=["servId", "error"])
df_errors.to_csv(ERROR_PATH, index=False, encoding="utf-8-sig")
print(f"요청 대상: {target_count:,}건 / 상세 확보: {len(df_detail):,}건 / 미확보: {target_count - len(df_detail):,}건")
print("오류 목록:", ERROR_PATH)

요청 대상: 1,485건 / 상세 확보: 1,000건 / 미확보: 485건
오류 목록: outputs\welfare_detail_errors.csv


## 10. 목록 + 상세 병합 → 최종 DataFrame

`df_all`의 목록 정보는 모두 보존하고, 상세조회 필드는 `detail_` 접두어를 붙여 병합합니다. 이렇게 해야 같은 필드명 충돌이나 누락된 상세 데이터에 의해 목록값이 덮어써지지 않습니다.

In [14]:
# 상세조회 ID는 문자열로 통일하고, 중복 ID가 있더라도 목록의 원래 행을 보존합니다.
df_all = df_all.copy()
df_all["servId"] = df_all["servId"].astype("string")

if df_detail.empty:
    raise RuntimeError("상세조회 성공 데이터가 없습니다. 상세조회 테스트 및 오류를 확인하세요.")

prefixed_detail = df_detail.copy()
prefixed_detail["servId"] = prefixed_detail["servId"].astype("string")
prefixed_detail = prefixed_detail.drop_duplicates(subset=["servId"], keep="last")
prefixed_detail = prefixed_detail.rename(
    columns={c: f"detail_{c}" for c in prefixed_detail.columns if c != "servId"}
)

df_full = df_all.merge(prefixed_detail, on="servId", how="left", validate="many_to_one")
df_full["detail_fetched"] = df_full["servId"].isin(set(df_detail["servId"].astype(str)))

df_jeonbuk_full = df_full[df_full["requested_region"] == "전북특별자치도"].copy()
df_seoul_full = df_full[df_full["requested_region"] == "서울특별시"].copy()
df_gyeonggi_full = df_full[df_full["requested_region"] == "경기도"].copy()

print("전체 목록 행 수:", len(df_all), "→ 병합 후:", len(df_full))
print("상세조회 성공 행 수:", int(df_full["detail_fetched"].sum()))
print("상세조회 누락 행 수:", int((~df_full["detail_fetched"]).sum()))

display(df_full[[c for c in [
    "servId", "servNm", "ctpvNm", "sggNm",
    "detail_sprtTrgtCn", "detail_slctCritCn", "detail_alwServCn", "detail_aplyMtdCn", "detail_fetched"
] if c in df_full.columns]].head(10))


전체 목록 행 수: 1485 → 병합 후: 1485
상세조회 성공 행 수: 1000
상세조회 누락 행 수: 485


,servId,servNm,ctpvNm,sggNm,detail_sprtTrgtCn,detail_slctCritCn,detail_alwServCn,detail_aplyMtdCn,detail_fetched
0,WLF00002438,군민 화장장려금 지원 사업,전북특별자치도,순창군,1. 사망일 현재 순창군에 연속하여 1년 이상 주민등록이 되어 있는 자가 사망하여 ...,순창군에 1년이상 주민등록이 되어있는 사람이 사망하여 화장의 방법으로 장례를 치른 ...,1구대 최대 50만원 화장장려금 지원,사망자의 관할 주소지 읍면사무소에 신청.[구비서류]1. 화장장려금 지원 신청서 1부...,True
1,WLF00001316,무료간병 온종일서비스,전북특별자치도,남원시,- 지원대상 : 기초생활수급자 및 차상위 중 1인가구 (연령제한 없음) / 긴급복지...,- 기초생활수급자 및 차상위 중 질병 발생시 간병비 부담이 어려워 치료를 포기하는 ...,□ 개별간병(1:1) - 최대 지원기간 14일(2주) - 지원기관 ① ...,- 지역거점 공공의료기관(남원의료원) 입원치료시 보건의료복지통합지원실을 통하여 상담...,True
2,WLF00003485,공공근로사업,전북특별자치도,무주군,1) 신청대상 - 신청일 현재 만18세 이상 만75세미만 무주군에 주소를 둔 자...,18세이상인자로서 가구중위소득 70%이하이면서 재산 보유 4억원 이하인 자 등,1) 서비스 지급내용\r - 실직자 및 저소득층 이들에게 한시적으로 일자리를 제공...,1) 신청방법 : 해당 읍ㆍ면에서 신청 2) 지급시기 및 지급방법 - 지급개시...,True
3,WLF00000453,행여자 관리(여비 및 숙박비 지급),전북특별자치도,정읍시,관내에서 발생한 행여자 및 무연고자,관내에서 발생한 행여자,"1) 행여자가 귀향여비를 신청할 경우 기차표 및 버스표 제공, 숙박비 제공\r2) ...",행여자가 직접 신청 (관할 경찰서 의뢰시 가능),True
4,WLF00002346,수급자가정 방문방역사업,전북특별자치도,임실군,기초생활수급자 및 차상위계층확인서 발급 가구,기초생활수급자 및 차상위계층 가구,"수급자가정 실내 및 주변 방역청소 : 기초수급자 주택이 자가인 경우(장기입원자, 장...",읍면사무소 사회복지담당,True
5,WLF00005117,부안군 아이돌봄서비스 본인부담금 지원,전북특별자치도,부안군,"신청일로부터 보호자와 아이가 부안군에 주민등록상 주소를 두고, 실제 거주하면서 아이...","- 신청일로부터 보호자와 아이가 부안군에 주민등록상 주소를 두고, 실제 거주하면서 ...","- 아이돌봄 지원사업 가, 나, 다, 라 유형 모두 본인부담금 70%지원",- 부안군 아이돌봄서비스 지원신청서 주소지 읍면장에게 제출- 구비서류(신청인 신분증...,True
6,WLF00005045,다문화가정 육아지원금 지원,전북특별자치도,임실군,□신청대상자-임실군에 거주하는 96개월 미만의 다문화가정 아동,□신청대상자-임실군에 거주하는 96개월 미만의 다문화가정 아동,-지급일: 매월 25일(지급일이 휴무일인 경우에는 그 전일에 지급)-지급액: 1인당...,□신청방법-신청자격이 되는 아동의 보호자가 읍·면사무소에서 신청서를 작성 후 제출,True
7,WLF00004579,여성농업인 생생카드 지원사업,전북특별자치도,NaN,실제 영농에 종사하고 농어촌지역에 거주하는 여성 농어업인으로 사업연도 기준 만20세...,실제 영농에 종사하고 농어촌지역에 거주하는 여성 농어업인으로 사업연도 기준 만20세...,1인 연간 지원액 : 13만원 지원내용 : 여성농어업인의 건강관리 및 문화·학습활동...,"사업신청을 받아 대상자 확정 후, 농협 채움카드 바우처 포인트 제공",True
8,WLF00002047,저소득 거동불편 장애인도시락배달,전북특별자치도,전주시,저소득 거동불편 장애인,등록장애인 중 저소득 가구이면서 거동이 불편한 장애인,도시락 배달,동 주민센터 신청 및 시행기관 신청,True
9,WLF00000489,보호대상아동 생활안정지원,전북특별자치도,NaN,"관내 가정위탁아동(보호연장, 종결 및 초중고)","관내 가정위탁아동(보호연장, 종결 및 초중고)",- 명절위로비 2회/세대당/100천원 - 방과후 간식비 인/180일/1천원 ...,"각 주소지 읍면, 완주군 아동친화과",True


## 11. 상세 포함 최종 CSV + JSONL 저장

JSONL은 한 줄에 서비스 1건을 저장하는 형식이며 RAG 전처리에 유용합니다. `detail_fetched=False`는 상세조회가 아직 되지 않은 항목이므로 완전 수집으로 오해하면 안 됩니다.

In [15]:
full_exports = {
    "jeonbuk_full": df_jeonbuk_full,
    "seoul_full": df_seoul_full,
    "gyeonggi_full": df_gyeonggi_full,
    "all_full": df_full,
    "detail_only": df_detail,
}

for name, data in full_exports.items():
    path = OUTPUT_DIR / f"welfare_{name}_{date_tag}.csv"
    data.to_csv(path, index=False, encoding="utf-8-sig")
    print("CSV 저장:", path)

jsonl_path = OUTPUT_DIR / f"welfare_all_full_{date_tag}.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for record in df_full.astype(object).where(pd.notna(df_full), None).to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
print("JSONL 저장:", jsonl_path)

missing_ids = sorted(set(df_all["servId"].dropna().astype(str)) - set(df_detail["servId"].astype(str)))
print(f"상세 미수집 서비스 ID: {len(missing_ids):,}건")
if missing_ids:
    print("미수집 ID 예시:", missing_ids[:10])


CSV 저장: outputs\welfare_jeonbuk_full_20261107_103452.csv
CSV 저장: outputs\welfare_seoul_full_20261107_103452.csv
CSV 저장: outputs\welfare_gyeonggi_full_20261107_103452.csv
CSV 저장: outputs\welfare_all_full_20261107_103452.csv
CSV 저장: outputs\welfare_detail_only_20261107_103452.csv
JSONL 저장: outputs\welfare_all_full_20261107_103452.jsonl
상세 미수집 서비스 ID: 485건
미수집 ID 예시: ['WLF00000001', 'WLF00000284', 'WLF00000286', 'WLF00000314', 'WLF00000315', 'WLF00000316', 'WLF00000572', 'WLF00000645', 'WLF00000649', 'WLF00000650']
